# Sequential vs. Parallel — Confidence Interval Analysis

This notebook reads the evaluation sheet (tabs **Sequential** and **Parallel**), keeps only questions that executed **successfully**, and for every number of devices computes:

- mean, standard deviation, n
- confidence interval: **mean ± t(1−α/2, n−1) · s / √n** (95% and 99%)
- whether the Sequential and Parallel intervals overlap

It then draws bar charts with error bars (the CI).

**Run the cells from top to bottom** (Runtime → Run all).

## 1. Settings
Choose where the data comes from:
- `SOURCE = "sheet"` reads your Google Sheet directly from its link (the sheet must be shared as *Anyone with the link can view*).
- `SOURCE = "upload"` lets you upload an `.xlsx` file from your computer.

To reproduce the thesis numbers, upload `Evaluation_Parallel_vs_Sequential.xlsx` from this folder.

In [ ]:
SOURCE = "upload"         # "upload" (xlsx from this repo) or "sheet" (Google Sheet link)
SHEET_URL = "PASTE_YOUR_GOOGLE_SHEET_LINK_HERE"

DEVICES = [2, 3, 4, 5]     # device counts to compare
LEVELS = [95, 99]          # confidence levels in %

## 2. Load the data

In [ ]:
import re, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

MODES = ["Sequential", "Parallel"]
COLORS = {"Sequential": "#2A78D6", "Parallel": "#EB6834"}
DEVICE_NAME = re.compile(r"sonic_4\.4\.0-\d+")

if SOURCE == "sheet":
    sheet_id = re.search(r"/d/([a-zA-Z0-9_-]+)", SHEET_URL).group(1)
    xlsx = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
else:
    from google.colab import files
    uploaded = files.upload()
    xlsx = io.BytesIO(next(iter(uploaded.values())))

sheets = pd.read_excel(xlsx, sheet_name=None)
print("Tabs found:", list(sheets))

In [ ]:
def prepare(df, mode):
    df = df[df["Question"].notna() & (df["Question"].astype(str).str.strip() != "")].copy()
    df["time"] = pd.to_numeric(df["Execution Time (s)"], errors="coerce")
    df["status"] = df["Execution Status"].astype(str).str.strip().str.lower()
    if "Devices" in df.columns and pd.to_numeric(df["Devices"], errors="coerce").notna().any():
        df["devices"] = pd.to_numeric(df["Devices"], errors="coerce")
    else:  # count the device names in the question
        df["devices"] = df["Question"].astype(str).apply(lambda q: len(set(DEVICE_NAME.findall(q))))
    df["mode"] = mode
    return df

all_rows = pd.concat([prepare(sheets[m], m) for m in MODES], ignore_index=True)
data = all_rows[(all_rows["status"] == "success") & (all_rows["time"] > 0)]

print("Successful questions per number of devices:")
display(pd.crosstab(data["devices"], data["mode"])[MODES])

## 3. Confidence intervals

In [ ]:
def ci(values, level):
    v = np.asarray(values, dtype=float)
    n = len(v)
    mean = v.mean() if n else np.nan
    if n < 2:
        return n, mean, np.nan, np.nan
    sd = v.std(ddof=1)
    margin = stats.t.ppf(1 - (1 - level) / 2, n - 1) * sd / np.sqrt(n)
    return n, mean, sd, margin

rows = []
for level in LEVELS:
    for d in DEVICES:
        for mode in MODES:
            n, mean, sd, margin = ci(data[(data["mode"] == mode) & (data["devices"] == d)]["time"], level / 100)
            rows.append({"Level (%)": level, "Devices": d, "Mode": mode, "n": n,
                         "Mean (s)": mean, "Std dev (s)": sd, "Margin ± (s)": margin,
                         "Lower (s)": mean - margin, "Upper (s)": mean + margin})
summary = pd.DataFrame(rows)

verdicts = []
for (level, d), g in summary.groupby(["Level (%)", "Devices"]):
    s = g[g["Mode"] == "Sequential"].iloc[0]
    p = g[g["Mode"] == "Parallel"].iloc[0]
    if s["Lower (s)"] > p["Upper (s)"]:
        result = "No overlap → Parallel is faster (significant)"
    elif p["Lower (s)"] > s["Upper (s)"]:
        result = "No overlap → Sequential is faster (significant)"
    else:
        result = "Overlap → cannot conclude"
    verdicts.append({"Level (%)": level, "Devices": d,
                     "Speedup (×)": s["Mean (s)"] / p["Mean (s)"], "Result": result})
verdicts = pd.DataFrame(verdicts)

for level in LEVELS:
    print(f"\n===== {level}% confidence interval =====")
    display(summary[summary["Level (%)"] == level].drop(columns="Level (%)").round(2).reset_index(drop=True))
    display(verdicts[verdicts["Level (%)"] == level].drop(columns="Level (%)").round(2).reset_index(drop=True))

## 4. Charts (mean with confidence-interval error bars)

In [ ]:
def plot(level):
    g = summary[summary["Level (%)"] == level]
    devs = sorted(g["Devices"].unique())
    x = np.arange(len(devs)); width = 0.38
    fig, ax = plt.subplots(figsize=(8, 5))
    for i, mode in enumerate(MODES):
        m = g[g["Mode"] == mode].set_index("Devices").loc[devs]
        bars = ax.bar(x + (i - 0.5) * width, m["Mean (s)"], width, yerr=m["Margin ± (s)"], capsize=6,
                      label=mode, color=COLORS[mode], error_kw={"elinewidth": 1.2, "ecolor": "#333333"})
        for bar, mean, margin in zip(bars, m["Mean (s)"], m["Margin ± (s)"]):
            ax.text(bar.get_x() + bar.get_width() / 2, mean + margin + 0.4, f"{mean:.1f}",
                    ha="center", va="bottom", fontsize=9, color="#333333")
    ax.set_xticks(x); ax.set_xticklabels([f"{d} devices" for d in devs])
    ax.set_xlabel("Number of target devices"); ax.set_ylabel("Mean execution time (s)")
    ax.set_title(f"Mean execution time with {level}% confidence intervals")
    ax.set_ylim(bottom=0); ax.yaxis.grid(True, color="#E6E6E6"); ax.set_axisbelow(True)
    for side in ("top", "right"): ax.spines[side].set_visible(False)
    ax.legend(frameon=False); fig.tight_layout()
    fig.savefig(f"ci_{level}.png", dpi=200)
    plt.show()

for level in LEVELS:
    plot(level)

## 5. Download the results (optional)
Saves the table as CSV and downloads it together with the chart images.

In [ ]:
summary.merge(verdicts, on=["Level (%)", "Devices"]).round(3).to_csv("ci_summary.csv", index=False)
from google.colab import files
for f in ["ci_summary.csv"] + [f"ci_{l}.png" for l in LEVELS]:
    files.download(f)